Pseudo-random procedurally generate MIDI sequences of specified degrees of polyphony and dynamics.

In [1]:
import random
import pandas as pd
import pretty_midi as pm

In [2]:
pitch_min = 21
pitch_max = 108

def sample_pitch(pitch_min, pitch_max):
    return random.randint(pitch_min, pitch_max)

duration_min = 0.01
duration_max = 5.

def sample_duration(duration_min, duration_max, alpha=2, beta=5):
    return random.betavariate(alpha, beta) * (duration_max-duration_min) + duration_min

dynamics_ranges = {
    0: {'velocity_min': 60, 'velocity_max': 68}, # easy - no dynamics (all mezzo-forte)
    1: {'velocity_min': 32, 'velocity_max': 96}, # medium - some dynamics (piano to forte)
    2: {'velocity_min': 1, 'velocity_max': 127}  # hard - full dynamics (piano pianissimo to forte fortissimo)
}

def sample_velocity(dynamics_range):
    return random.randint(dynamics_range['velocity_min'], dynamics_range['velocity_max'])

polyphony_min = 1
polyphony_max = 24

polyphony_degrees = list(range(polyphony_min, polyphony_max+1))

D = len(dynamics_ranges) # dynamics ranges
P = len(polyphony_degrees) # number of polyphony degrees to be explored
T = 2 # minutes per piece
print('| P =', P, '| T =', T, '| D =', D, '| P * T * D =', P * T * D, 'minutes.')

piece_duration = T * 60. # 120 seconds

| P = 24 | T = 2 | D = 3 | P * T * D = 144 minutes.


In [3]:
def overlaps(note_a, note_b):
    return (min(note_a.end, note_b.end) - max(note_a.start, note_b.start)) > 0

def generate_random_piece(polyphony_deg, dynamics_range, piece_duration, discontinuous_voices=True, seed=0):
    assert polyphony_deg > 0
    random.seed(seed)

    # create a PrettyMIDI object for the piece
    piece = pm.PrettyMIDI(resolution=480)
    # create an Instrument instance for the piano instrument
    piano_program = pm.instrument_name_to_program('Acoustic Grand Piano') # 0
    piano = pm.Instrument(program=piano_program)

    # for each voice of the polyphony
    for voice in range(polyphony_deg):
        # populate the voice with notes
        t_playhead = 0.
        while t_playhead < piece_duration:
            # keep resampling the new note until one is found that doesn't overlap
            # with any of the existing notes
            while True:
                velocity = sample_velocity(dynamics_range)
                pitch = sample_pitch(pitch_min, pitch_max)
                duration = sample_duration(duration_min, duration_max)
                new_note = pm.Note(velocity=velocity, pitch=pitch, start=t_playhead, end=t_playhead+duration)

                equal_pitch_notes = [note for note in piano.notes if note.pitch == new_note.pitch]
                if not any(overlaps(new_note, note) for note in equal_pitch_notes):
                    break
                else:
                    # TODO in script as logging.log(logging.INFO, text)
                    print(f'collision in voice {voice}: pitch {pitch} at time {new_note.start:.2f}-{new_note.end:.2f} [s] overlaps. resampling...')
            
            # trim the duration of the new note, if it would extend beyond the piece duration
            if t_playhead + duration > piece_duration:
                duration = piece_duration - t_playhead
                new_note.end = t_playhead + duration
                t_playhead = piece_duration # ensure breaking the loop for this voice (prevent float precision issues)
            else:
                t_playhead += duration
            
            # add the new note to the piano instrument
            piano.notes.append(new_note)
    
    if discontinuous_voices:
        # randomize discontinuity of voices by trimming the notes by 
        # up to 1% of their durations on both start and end times
        for note in piano.notes[1:-1]:
            # note.duration changes together with note.start and note.end
            note_duration = note.duration
            note.start += random.uniform(0.0, 0.01 * note_duration)
            note.end -= random.uniform(0.0, 0.01 * note_duration)
            
    # add the piano instrument to the piece
    piece.instruments.append(piano)
    return piece

# test execution
piece = generate_random_piece(1, dynamics_ranges[0], piece_duration, seed=0)

Synthesize the MIDI files and curate metadata CSV

In [ ]:
!mkdir -p ../data/2_random/

records_list = []
for polyphony_deg in polyphony_degrees:
    for dyn_level in dynamics_ranges.keys():
        config_string = f'RANDOM_P{polyphony_deg:02d}_D{dyn_level:01d}'
        piece = generate_random_piece(polyphony_deg, dynamics_ranges[dyn_level], piece_duration, seed=0)
        # piece.write(f'../data/2_random/{config_string}.mid')

        # num_notes
        num_notes = len(piece.instruments[0].notes)
        # num_unique_pitches
        unique_pitches = set(note.pitch for note in piece.instruments[0].notes)
        num_unique_pitches = len(unique_pitches)
        # num_unique_velocities
        unique_velocities = set(note.velocity for note in piece.instruments[0].notes)
        num_unique_velocities = len(unique_velocities)

        new_record = {'mds_file': config_string, 'polyphony': polyphony_deg, 'dynamics': dyn_level,
                      'num_notes': num_notes, 'num_unique_pitches': num_unique_pitches, 'num_unique_velocities': num_unique_velocities}
        records_list.append(new_record)

df = pd.DataFrame.from_records(records_list)
# df.to_csv('../data/metadata/2_random_meta.csv', index=False)
df

,mds_file,polyphony,dynamics,num_notes,num_unique_pitches,num_unique_velocities
0,RANDOM_P01_D0,1,0,84,54,9
1,RANDOM_P01_D1,1,1,85,60,46
2,RANDOM_P01_D2,1,2,79,52,57
3,RANDOM_P02_D0,2,0,164,77,9
4,RANDOM_P02_D1,2,1,169,76,58
...,...,...,...,...,...,...
67,RANDOM_P23_D1,23,1,2077,88,65
68,RANDOM_P23_D2,23,2,2056,88,127
69,RANDOM_P24_D0,24,0,2160,88,9
70,RANDOM_P24_D1,24,1,2158,88,65
